The goal of this project is to create a blackjack solver. For each hand versus the dealer's upcard, the solver will determine the optimal action.
First of all lets start with implementing the basic hit or stand. Aces are going to count as 1 always for now. We will implement the other actions later.

In [66]:
# general q-learning algorithm

from collections import defaultdict
from random import random
from typing import Callable



def train_q_learning(
    q_learning_episode : Callable[[dict, float, float, float], None],
    episodes=1000000,
    learning_rate=0.1,
    discount_factor=1.0,
    start_epsilon=1.0,
    end_epsilon=0.05,
):
    q_table = defaultdict(lambda: {action: 0.0 for action in ACTIONS})

    for episode in range(episodes):
        epsilon = max(
            end_epsilon,
            start_epsilon - (start_epsilon - end_epsilon) * episode / episodes,
        )

        q_learning_episode(q_table, learning_rate, discount_factor, epsilon)

    return q_table



def choose_training_action(q_table, state, epsilon):
    if random.random() < epsilon:
        return random.choice(list(ACTIONS))
    return max(ACTIONS, key=lambda action: q_table[state][action])

In [67]:
# general blackjack algorithms

import random
from enum import Enum, auto

class ACTIONS(Enum):
    STAND = auto()
    HIT = auto()

CARDS = [2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 10, 10, 11]  # 11 represents an Ace


def draw_card():
    return random.choice(CARDS)


def dealer_total(dealer_upcard):
    ace = dealer_upcard == 11
    total = dealer_upcard
    while total < 17:
        card = draw_card()
        total += card

        if card == 11:
            if total > 21:
                total -= 10
            else:
                ace = True

        if total > 21 and ace:
            total -= 10
            ace = False
    return total


def reward_for_standing(player_total, dealer_final_total):
    if dealer_final_total > 21 or player_total > dealer_final_total:
        return 1.0
    if player_total < dealer_final_total:
        return -1.0
    return 0.0



In [68]:
import pandas as pd


def print_table(q_table, state_columns):
    rows = []
    for state, action_values in sorted(q_table.items()):
        row = dict(zip(state_columns, state))
        row.update({action.name: value for action, value in action_values.items()})
        row["Best Action"] = max(action_values, key=action_values.get).name
        rows.append(row)

    table = pd.DataFrame(rows)
    print(table.to_string(index=False, float_format="{:.2f}".format))

In [69]:
def q_learning_episode(q_table, learning_rate, discount_factor, epsilon):
    ace = False
    card1 = draw_card()
    card2 = draw_card()
    player_total = card1 + card2
    if card1 == 11 or card2 == 11:
        player_total = card1 + card2
        ace = True
        if card1 == 11 and card2 == 11:
            player_total = 12

    dealer_upcard = draw_card()
    while True:
        state = (player_total, dealer_upcard, ace)
        action = choose_training_action(q_table, state, epsilon)

        if action == ACTIONS.STAND:
            reward = reward_for_standing(
                            player_total,
                            dealer_total(dealer_upcard),
                        )
            next_value = 0.0
            done = True
            
        elif action == ACTIONS.HIT:
            card = draw_card()
            player_total += card
            if card == 11:
                if player_total > 21:
                    player_total -= 10
                else:
                    ace = True

            if player_total > 21 and ace:
                player_total -= 10
                ace = False

            if player_total > 21:
                reward = -1.0
                next_value = 0.0
                done = True
            else:
                reward = 0.0
                next_state = (player_total, dealer_upcard, ace)
                next_value = max(q_table[next_state].values())
                done = False

        old_value = q_table[state][action]
        target = reward + discount_factor * next_value
        q_table[state][action] = old_value + learning_rate * (target - old_value)

        if done:
            break

In [70]:
random.seed(42)
trained_q_table = train_q_learning(q_learning_episode)

print_table(trained_q_table, state_columns=["Player Total", "Dealer Upcard", "Ace"])

 Player Total  Dealer Upcard   Ace  STAND   HIT Best Action
            4              2 False  -0.38 -0.10         HIT
            4              3 False  -0.24 -0.10         HIT
            4              4 False  -0.16 -0.13         HIT
            4              5 False  -0.31 -0.09         HIT
            4              6 False  -0.13 -0.08         HIT
            4              7 False  -0.66 -0.06         HIT
            4              8 False  -0.66 -0.21         HIT
            4              9 False  -0.48 -0.12         HIT
            4             10 False  -0.69 -0.35         HIT
            4             11 False  -0.82 -0.47         HIT
            5              2 False  -0.56 -0.25         HIT
            5              3 False  -0.57 -0.15         HIT
            5              4 False  -0.48 -0.17         HIT
            5              5 False  -0.17  0.01         HIT
            5              6 False  -0.50 -0.14         HIT
            5              7 False  -0.5

In [71]:
from blackjack_solver import train_q_learning

trained_q_table_with_visit_counts, visit_counts = train_q_learning(seed=42)

print_table(
    trained_q_table_with_visit_counts,
    state_columns=["Player Total", "Dealer Upcard", "Ace"],
)

print("\nVisit Counts:")
print_table(
    visit_counts,
    state_columns=["Player Total", "Dealer Upcard", "Ace"],
)

 Player Total  Dealer Upcard   Ace  STAND   HIT Best Action
            4              2 False  -0.52 -0.12         HIT
            4              3 False  -0.41 -0.08         HIT
            4              4 False  -0.23 -0.03         HIT
            4              5 False  -0.16 -0.01         HIT
            4              6 False  -0.15  0.01         HIT
            4              7 False  -0.56 -0.09         HIT
            4              8 False  -0.65 -0.17         HIT
            4              9 False  -0.60 -0.25         HIT
            4             10 False  -0.58 -0.33         HIT
            4             11 False  -0.68 -0.44         HIT
            5              2 False  -0.48 -0.15         HIT
            5              3 False  -0.30 -0.11         HIT
            5              4 False  -0.23 -0.05         HIT
            5              5 False  -0.17 -0.01         HIT
            5              6 False  -0.04  0.01         HIT
            5              7 False  -0.4